In [0]:
bronze_path = "abfss://bronze@netflixxdata.dfs.core.windows.net/"
 
files = dbutils.fs.ls(bronze_path)
 
for file in files:
    print(file.name, "Folder:", file.isDir())

netflix_cast/ Folder: True
netflix_category/ Folder: True
netflix_countries/ Folder: True
netflix_directors/ Folder: True
netflix_titles/ Folder: True


In [0]:

bronze_path = "abfss://bronze@netflixxdata.dfs.core.windows.net/"
 
for folder in dbutils.fs.ls(bronze_path):
    if folder.isDir() and "titles" not in folder.name.lower():
        print("FOLDER:", folder.name)
 
        for file in dbutils.fs.ls(folder.path):
            print("  FILE:", file.name)

FOLDER: netflix_cast/
  FILE: netflix_cast.csv
FOLDER: netflix_category/
  FILE: netflix_category.csv
FOLDER: netflix_countries/
  FILE: netflix_countries.csv
FOLDER: netflix_directors/
  FILE: netflix_directors.csv


In [0]:
titles_path = "abfss://bronze@netflixxdata.dfs.core.windows.net/netflix_titles"
 
df_titles = spark.read.format("delta").load(titles_path)
 
print("Netflix Titles records:", df_titles.count())

Netflix Titles records: 6236


In [0]:
# Create two parameters
dbutils.widgets.text("folder_name", "netflix_cast")
dbutils.widgets.text("file_name", "netflix_cast.csv")
 
# Retrieve their values
folder_name = dbutils.widgets.get("folder_name")
file_name = dbutils.widgets.get("file_name")
 
print("Folder:", folder_name)
print("File:", file_name)

Folder: netflix_cast
File: netflix_cast.csv


In [0]:
# Construct the source path using parameters
source_path = f"abfss://bronze@netflixxdata.dfs.core.windows.net/{folder_name}/{file_name}"
 
# Read the CSV file
df_lookup = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)
 
print("Reading:", source_path)
print("Total records:", df_lookup.count())

Reading: abfss://bronze@netflixxdata.dfs.core.windows.net/netflix_cast/netflix_cast.csv
Total records: 44311


In [0]:
# Construct the Silver destination path
target_path = f"abfss://silver@netflixxdata.dfs.core.windows.net/{folder_name}"
 
# Save the data as Delta
(
    df_lookup.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(target_path)
)
 
print("Successfully saved to:", target_path)

Successfully saved to: abfss://silver@netflixxdata.dfs.core.windows.net/netflix_cast
